In [9]:
import json
from pathlib import Path

DATA_DIR = Path.cwd()
jsonl_files = sorted(DATA_DIR.glob('nas_trials_*.jsonl'))
if not jsonl_files:
    print('No nas_trials_*.jsonl files found in output directory')
else:
    def format_layer(layer: dict) -> str:
        layer_type = layer.get('type')
        if layer_type == 'Conv1d':
            return (
                f"Conv1d(in={layer['in_channels']}, out={layer['out_channels']}, "
                f"k={layer['kernel_size']}, s={layer['stride']}, p={layer['padding']})"
            )
        if layer_type == 'MaxPool1d':
            return (
                f"MaxPool1d(k={layer['kernel_size']}, s={layer['stride']}, p={layer['padding']})"
            )
        if layer_type == 'Linear':
            bias = 'yes' if layer.get('bias') else 'no'
            return f"Linear(in={layer['in_features']}, out={layer['out_features']}, bias={bias})"
        if layer_type == 'Flatten':
            return f"Flatten(start={layer['start_dim']}, end={layer['end_dim']})"
        if layer_type == 'ELU':
            return f"ELU(alpha={layer.get('alpha', 1.0)})"
        if layer_type == 'ReLU':
            return "ReLU()"
        return str(layer)

    latest = jsonl_files[-1]
    print(f"Loaded {latest.name}")
    for line in latest.read_text().splitlines():
        if not line.strip():
            continue
        record = json.loads(line)
        trial = record['trial_number']
        rmse = record['rmse']
        print(f"\nTrial {trial} — RMSE: {rmse:.4f}")
        target = record['target_col']
        arch = record['arch_id']
        activation = record['activation']
        print(f"Target: {target}  Arch: {arch}  Activation: {activation}")
        for layer in record.get('layers', []):
            print(f"    {format_layer(layer)}")


Loaded nas_trials_20260404T004436.jsonl

Trial 0 — RMSE: 0.1065
Target: left_wall_dist  Arch: 8  Activation: relu
    Conv1d(in=1, out=16, k=3, s=1, p=0)
    ReLU()
    MaxPool1d(k=2, s=2, p=0)
    Conv1d(in=16, out=17, k=3, s=1, p=0)
    ReLU()
    MaxPool1d(k=2, s=2, p=0)
    Conv1d(in=17, out=36, k=3, s=1, p=0)
    ReLU()
    MaxPool1d(k=2, s=2, p=0)
    Conv1d(in=36, out=117, k=3, s=1, p=0)
    ReLU()
    MaxPool1d(k=8, s=8, p=0)
    Flatten(start=1, end=-1)
    Linear(in=1872, out=103, bias=yes)
    ReLU()
    Linear(in=103, out=1, bias=yes)

Trial 1 — RMSE: 0.1010
Target: left_wall_dist  Arch: 8  Activation: elu
    Conv1d(in=1, out=52, k=3, s=1, p=0)
    ELU(alpha=1.0)
    MaxPool1d(k=2, s=2, p=0)
    Conv1d(in=52, out=24, k=3, s=1, p=0)
    ELU(alpha=1.0)
    MaxPool1d(k=4, s=4, p=0)
    Conv1d(in=24, out=82, k=3, s=1, p=0)
    ELU(alpha=1.0)
    MaxPool1d(k=2, s=2, p=0)
    Conv1d(in=82, out=19, k=3, s=1, p=0)
    ELU(alpha=1.0)
    MaxPool1d(k=2, s=2, p=0)
    Conv1d(in=19, o